# Predicting Final State with Long Short‐Term Memory Neural Networks


Other papers: 

- Compares performance of traditional time series model to Neural Networks:
     Lou, H.-R., Wang, X., Gao, Y., & Zeng, Q. (2022). Comparison of ARIMA model, DNN model and LSTM model in predicting disease burden of occupational pneumoconiosis in Tianjin, China. BMC Public Health, 22(1), Article 2167. https://doi.org/10.1186/s12889-022-14642-3

- Predicting Parkisons Diagnosis:

  Tekindor, A. N., & Akman Aydın, E. (2025). Speech signals-based Parkinson’s disease diagnosis using hybrid autoencoder-LSTM models. Computers in Biology and Medicine, 193, Article 110334. https://doi.org/10.1016/j.compbiomed.2025.110334


In [49]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import cross_val_predict
import random

In [ ]:

# Load the data
data = pd.read_csv('sledata_processed.csv', parse_dates=['ASSDT'])

# Sort by patient and assessment date
data = data.sort_values(['PTNO', 'ASSDT'])

# Calculate time differences
data['time_since_first'] = data.groupby('PTNO')['ASSDT'].transform(
    lambda x: (x - x.min()).dt.days
)
data['time_since_last'] = data.groupby('PTNO')['ASSDT'].transform(
    lambda x: x.diff().dt.days.fillna(0)
)

# Convert categorical EMPf to numerical
emp_mapping = {v: k for k, v in enumerate(data['EMPf'].unique())}
data['EMP_numeric'] = data['EMPf'].map(emp_mapping)

# Encode the target (end_state)
label_encoder = LabelEncoder()
data['end_state_encoded'] = label_encoder.fit_transform(data['end_state'])

# Features to use
features = ['EMP_numeric', 'SLEDAI2_I', 'score_n', 'STERDOSE', 'ISDOSE', 'INCEPT', 'STERUSE',
            'AMDOSE', 'age_at_record', 'time_since_last', 
            'time_since_first', 'visit_num']

# Identify categorical columns (non-numeric)
categorical_cols = data[features].select_dtypes(include=['object', 'category']).columns

# Label encode categorical columns
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))


# Fill NA values
data[features] = data[features].fillna(0)

# Normalize features
scaler = MinMaxScaler()
data[features] = scaler.fit_transform(data[features])

# Create sequences for each patient
def create_sequences(data, features, max_seq_length=None):
    patients = data['PTNO'].unique()
    num_features = len(features)
    
    if not max_seq_length:
        max_seq_length = data.groupby('PTNO').size().max()
    
    X = np.zeros((len(patients), max_seq_length, num_features))
    y = np.zeros(len(patients))
    seq_lengths = np.zeros(len(patients))
    
    for i, patient in enumerate(patients):
        patient_data = data[data['PTNO'] == patient].sort_values('ASSDT')
        seq_len = len(patient_data)
        seq_lengths[i] = seq_len
        
        # Fill the sequence data
        X[i, :seq_len, :] = patient_data[features].values
        
        # Get the end_state label (last record)
        y[i] = patient_data['end_state_encoded'].iloc[-1]
    
    return X, to_categorical(y), seq_lengths

# Create sequences
X, y, seq_lengths = create_sequences(data, features)

# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Build LSTM model
num_classes = y.shape[1]
model = Sequential([
    Masking(mask_value=0., input_shape=(X.shape[1], X.shape[2])),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)


/var/folders/x9/3zl22m596tl6d44cbs62z6h00000gn/T/ipykernel_92007/811433957.py:2: DtypeWarning: Columns (125,127,134,136,143,145,150,180) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('sledata_processed.csv', parse_dates=['ASSDT'])


## Train Test Split

In [40]:
random.seed(365)
# Create sequences for each patient
def create_sequences(data, features, max_seq_length=None):
    patients = data['PTNO'].unique()
    num_features = len(features)
    
    if not max_seq_length:
        max_seq_length = data.groupby('PTNO').size().max()
    
    X = np.zeros((len(patients), max_seq_length, num_features))
    y = np.zeros(len(patients))
    seq_lengths = np.zeros(len(patients))
    
    for i, patient in enumerate(patients):
        patient_data = data[data['PTNO'] == patient].sort_values('ASSDT')
        seq_len = len(patient_data)
        seq_lengths[i] = seq_len
        
        # Fill the sequence data
        X[i, :seq_len, :] = patient_data[features].values
        
        # Get the end_state label (last record)
        y[i] = patient_data['end_state_encoded'].iloc[-1]
    
    return X, to_categorical(y), seq_lengths



# Create sequences
X, y, seq_lengths = create_sequences(data, features)

# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Build LSTM model
num_classes = y.shape[1]
model = Sequential([
    Masking(mask_value=0., input_shape=(X.shape[1], X.shape[2])),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# Early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [42]:
random.seed(365)
# Corrected model training section
from sklearn.utils.class_weight import compute_class_weight

# Calculate class weights
y_integers = np.argmax(y_train, axis=1)
class_weights = compute_class_weight('balanced', classes=np.unique(y_integers), y=y_integers)
class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

# Train the model with corrected class_weight
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=50,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    class_weight=class_weight_dict  # Use the computed weights
)

# Rest of your code remains the same...

# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {test_acc:.4f}')

# Predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_true_classes, 
    y_pred_classes, 
    target_names=label_encoder.classes_
))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_true_classes, y_pred_classes))

Epoch 1/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 198ms/step - accuracy: 0.6419 - loss: 0.8502 - val_accuracy: 0.5889 - val_loss: 1.1107
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 187ms/step - accuracy: 0.6288 - loss: 0.5859 - val_accuracy: 0.6167 - val_loss: 0.9877
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 155ms/step - accuracy: 0.6632 - loss: 0.6084 - val_accuracy: 0.5944 - val_loss: 1.0116
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 145ms/step - accuracy: 0.6362 - loss: 0.6933 - val_accuracy: 0.6000 - val_loss: 1.0111
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 217ms/step - accuracy: 0.6708 - loss: 0.6174 - val_accuracy: 0.6500 - val_loss: 0.9656
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 151ms/step - accuracy: 0.6710 - loss: 0.6000 - val_accuracy: 0.6500 - val_loss: 0.9355
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 182ms/step - accuracy: 0.6818 - loss: 0.5479 - val_accuracy: 0.6278 - val_loss: 0.9677
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 6s 275ms/step - accuracy: 0.6696 - loss: 0.5167 - val_accuracy: 0.

## Cross-Validation

In [47]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, Masking
from keras.optimizers import Adam
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
import random

# Set random seeds for reproducibility
random.seed(365)
np.random.seed(365)

# Create sequences for each patient (your existing function)
def create_sequences(data, features, max_seq_length=None):
    patients = data['PTNO'].unique()
    num_features = len(features)
    
    if not max_seq_length:
        max_seq_length = data.groupby('PTNO').size().max()
    
    X = np.zeros((len(patients), max_seq_length, num_features))
    y = np.zeros(len(patients))
    seq_lengths = np.zeros(len(patients))
    
    for i, patient in enumerate(patients):
        patient_data = data[data['PTNO'] == patient].sort_values('ASSDT')
        seq_len = len(patient_data)
        seq_lengths[i] = seq_len
        
        # Fill the sequence data
        X[i, :seq_len, :] = patient_data[features].values
        
        # Get the end_state label (last record)
        y[i] = patient_data['end_state_encoded'].iloc[-1]
    
    return X, to_categorical(y), seq_lengths

# Create sequences
X, y, seq_lengths = create_sequences(data, features)

# Convert one-hot encoded y back to integers for stratified k-fold
y_integers = np.argmax(y, axis=1)

# Initialize Stratified K-Fold cross-validator
n_splits = 5  # You can adjust this number
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Store results from each fold
fold_results = []
conf_matrices = []
classification_reports = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y_integers)):
    print(f"\n=== Fold {fold + 1}/{n_splits} ===")
    
    # Split data
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # Calculate class weights for this fold
    y_train_integers = np.argmax(y_train, axis=1)
    class_weights = compute_class_weight('balanced', 
                                       classes=np.unique(y_train_integers), 
                                       y=y_train_integers)
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    
    # Build LSTM model (same architecture for each fold)
    num_classes = y.shape[1]
    model = Sequential([
        Masking(mask_value=0., input_shape=(X.shape[1], X.shape[2])),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ])
    
    # Compile the model
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=0.001),
        metrics=['accuracy']
    )
    
    # Early stopping callback
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True
    )
    
    # Train the model
    history = model.fit(
        X_train, y_train,
        batch_size=32,
        epochs=50,
        validation_data=(X_test, y_test),
        callbacks=[early_stopping],
        class_weight=class_weight_dict,
        verbose=1
    )
    
    # Evaluate the model
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    fold_results.append(test_acc)
    
    # Predictions
    y_pred = model.predict(X_test, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)
    
    # Store metrics
    conf_matrices.append(confusion_matrix(y_true_classes, y_pred_classes))
    classification_reports.append(classification_report(
        y_true_classes, 
        y_pred_classes, 
        target_names=label_encoder.classes_
    ))
    
    print(f"Fold {fold + 1} Accuracy: {test_acc:.4f}")

# Print overall cross-validation results
print("\n=== Cross-Validation Results ===")
print(f"Average Accuracy: {np.mean(fold_results):.4f} (±{np.std(fold_results):.4f})")
print("\nIndividual Fold Accuracies:")
for i, acc in enumerate(fold_results):
    print(f"Fold {i + 1}: {acc:.4f}")

# You can also analyze the confusion matrices and classification reports
# from each fold if needed


=== Fold 1/5 ===


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 203ms/step - accuracy: 0.1551 - loss: 1.9291 - val_accuracy: 0.2389 - val_loss: 1.7854
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 157ms/step - accuracy: 0.3143 - loss: 1.6711 - val_accuracy: 0.1500 - val_loss: 1.8534
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 144ms/step - accuracy: 0.2214 - loss: 1.5194 - val_accuracy: 0.2111 - val_loss: 1.6919
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 161ms/step - accuracy: 0.3620 - loss: 1.5627 - val_accuracy: 0.4778 - val_loss: 1.6014
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 150ms/step - accuracy: 0.3920 - loss: 1.8053 - val_accuracy: 0.4056 - val_loss: 1.6528
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 141ms/step - accuracy: 0.3911 - loss: 1.2557 - val_accuracy: 0.2722 - val_loss: 1.7038
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 141ms/step - accuracy: 0.3471 - loss: 1.2332 - val_accuracy: 0.1889 - val_loss: 1.5141
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 145ms/step - accuracy: 0.3050 - loss: 1.3669 - val_accuracy: 0.

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


23/23 ━━━━━━━━━━━━━━━━━━━━ 11s 212ms/step - accuracy: 0.2142 - loss: 1.7703 - val_accuracy: 0.5000 - val_loss: 1.7670
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 183ms/step - accuracy: 0.4715 - loss: 1.7906 - val_accuracy: 0.5611 - val_loss: 1.7390
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 162ms/step - accuracy: 0.4972 - loss: 1.6387 - val_accuracy: 0.7611 - val_loss: 1.6046
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 121ms/step - accuracy: 0.5385 - loss: 1.4180 - val_accuracy: 0.4278 - val_loss: 1.5638
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 127ms/step - accuracy: 0.3351 - loss: 1.4723 - val_accuracy: 0.2167 - val_loss: 1.5587
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 128ms/step - accuracy: 0.3138 - loss: 1.4462 - val_accuracy: 0.1222 - val_loss: 1.5140
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 129ms/step - accuracy: 0.2445 - loss: 1.3670 - val_accuracy: 0.1333 - val_loss: 1.5169
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 119ms/step - accuracy: 0.2169 - loss: 1.4254 - val_accuracy: 0.1611 - val

...

23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 178ms/step - accuracy: 0.1964 - loss: 2.0234 - val_accuracy: 0.0500 - val_loss: 1.8167
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 145ms/step - accuracy: 0.0904 - loss: 1.8204 - val_accuracy: 0.0722 - val_loss: 1.7749
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 124ms/step - accuracy: 0.1489 - loss: 1.6657 - val_accuracy: 0.0556 - val_loss: 1.6901
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 131ms/step - accuracy: 0.1246 - loss: 1.6512 - val_accuracy: 0.0944 - val_loss: 1.6813
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 154ms/step - accuracy: 0.1557 - loss: 1.5221 - val_accuracy: 0.1333 - val_loss: 1.5247
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 154ms/step - accuracy: 0.1591 - loss: 1.3964 - val_accuracy: 0.1556 - val_loss: 1.4685
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 122ms/step - accuracy: 0.2868 - loss: 1.3321 - val_accuracy: 0.1778 - val_loss: 1.3930
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 157ms/step - accuracy: 0.3258 - loss: 1.1861 - val_accuracy: 0.2444 - val

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


23/23 ━━━━━━━━━━━━━━━━━━━━ 9s 206ms/step - accuracy: 0.1874 - loss: 1.8791 - val_accuracy: 0.6778 - val_loss: 1.7084
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 158ms/step - accuracy: 0.6177 - loss: 1.5660 - val_accuracy: 0.7389 - val_loss: 1.6030
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 167ms/step - accuracy: 0.6802 - loss: 1.5148 - val_accuracy: 0.7778 - val_loss: 1.5496
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 189ms/step - accuracy: 0.6688 - loss: 1.4051 - val_accuracy: 0.8333 - val_loss: 1.4484
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 182ms/step - accuracy: 0.6374 - loss: 1.4887 - val_accuracy: 0.7611 - val_loss: 1.4239
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 132ms/step - accuracy: 0.6526 - loss: 1.4797 - val_accuracy: 0.7500 - val_loss: 1.4059
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 139ms/step - accuracy: 0.6075 - loss: 1.5727 - val_accuracy: 0.8722 - val_loss: 1.1835
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 133ms/step - accuracy: 0.6690 - loss: 1.3870 - val_accuracy: 0.6722 - val_

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/core/masking.py:48: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 177ms/step - accuracy: 0.0515 - loss: 1.6593 - val_accuracy: 0.2011 - val_loss: 1.8277
Epoch 2/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 158ms/step - accuracy: 0.1189 - loss: 1.6025 - val_accuracy: 0.1732 - val_loss: 1.7855
Epoch 3/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 139ms/step - accuracy: 0.1107 - loss: 1.7942 - val_accuracy: 0.1285 - val_loss: 1.8032
Epoch 4/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 141ms/step - accuracy: 0.1651 - loss: 1.6665 - val_accuracy: 0.2067 - val_loss: 1.7643
Epoch 5/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 151ms/step - accuracy: 0.2133 - loss: 1.4449 - val_accuracy: 0.2905 - val_loss: 1.6669
Epoch 6/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 179ms/step - accuracy: 0.2766 - loss: 1.3863 - val_accuracy: 0.3520 - val_loss: 1.5834
Epoch 7/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 150ms/step - accuracy: 0.3745 - loss: 1.3168 - val_accuracy: 0.2793 - val_loss: 1.5863
Epoch 8/50
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 156ms/step - accuracy: 0.3959 - loss: 1.2540 - val_accuracy: 0.6872 - val_

In [48]:
# Calculate overall metrics
overall_accuracy = np.mean(fold_results)
std_accuracy = np.std(fold_results)

print("\n=== Final Cross-Validation Results ===")
print(f"Overall Accuracy: {overall_accuracy:.4f} (±{std_accuracy:.4f})")
print("\nDetailed Fold Accuracies:")
for i, acc in enumerate(fold_results, 1):
    print(f"Fold {i}: {acc:.4f}")


=== Final Cross-Validation Results ===
Overall Accuracy: 0.7609 (±0.0240)

Detailed Fold Accuracies:
Fold 1: 0.7500
Fold 2: 0.7500
Fold 3: 0.7278
Fold 4: 0.7833
Fold 5: 0.7933
